# Cloudflare AI Gateway ↔ Lago

Two things, both landing on the same `llm_cost` metric — breakable by model in
Lago because every event already carries `model` in its properties, and the
plan's `llm_cost` charge has `grouped_by: ["model"]` set:

1. **Backfill** — read every log entry from your Cloudflare AI Gateway and
   bill each one's *real*, already-metered cost straight from Cloudflare's
   own `cost` field. Idempotent: safe to re-run over the same window.
2. **Live call** — wrap a real provider SDK client, make one call through the
   gateway, and let `sdk.wrap()` bill it automatically.

### Setup

Set these as environment variables before starting the kernel (never hardcode
real credentials into the notebook itself) — **or** put them in a `.env` file
next to this notebook (`examples/.env`); the next cell loads one automatically
if present. A `.env` file survives kernel restarts, unlike shell exports made
after Jupyter is already running — if you restart the kernel and still see a
missing-variable error, that's usually why.

| Variable | What it is |
|---|---|
| `CF_ACCOUNT_ID` | Cloudflare account id |
| `CF_GATEWAY_ID` | the AI Gateway's id |
| `CF_LOGS_TOKEN` | Cloudflare API token scoped for AI Gateway logs read |
| `CF_GATEWAY_AUTH` | the gateway's own auth token (`cf-aig-authorization`) |
| `LAGO_API_KEY` | your Lago API key |
| `LAGO_API_URL` | defaults to `https://api.getlago.com/api/v1` |
| `LAGO_SUBSCRIPTION_ID` | defaults to `cloudflare_gateway_demo_sub` |
| `LAGO_VERIFY_SSL` | defaults to `true`. Set to `false` **only** for a local dev Lago instance behind a self-signed certificate — never for a real Lago URL. Lets you hit a local instance directly instead of needing a public tunnel just to get a browser-trusted cert. |
| `ANTHROPIC_API_KEY` / `MISTRAL_API_KEY` | only needed for the provider you pick in Part 2 — `workers-ai` needs none at all, it's billed directly by Cloudflare. `MISTRAL_API_KEY` doubles as pricing's alias-resolution credential (see below) — without it, a `-latest` Mistral call still bills, just as an unpriced token-event fallback. |

Mistral has no per-token price table of its own, so `mistral-small-latest`
(what you actually call) can't be priced directly. Setting `MISTRAL_API_KEY`
lets price mode resolve it via Mistral's own `/v1/models` — which reports
`mistral-small-latest`'s real dated id (`mistral-small-2603`) — and look
*that* up against OpenRouter, which does list it with real pricing.

In [ ]:
%load_ext autoreload
%autoreload 2
# Reloads lago_agent_sdk automatically whenever its source changes, so fixes
# take effect on the next cell run — no kernel restart needed. Only helps for
# edits made AFTER this cell has run once in the current kernel; the first
# time you pull in a change to this cell itself (or add a brand new
# top-level name the rest of the notebook needs), you still need one restart.

import os
import sys

import requests

sys.path.insert(0, "../src")  # run this notebook from examples/, or adjust to your install


def _load_dotenv(path: str) -> None:
    """No extra dependency — just KEY=VALUE lines, same as python-dotenv's basics."""
    if not os.path.exists(path):
        return
    for line in open(path):
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, _, value = line.partition("=")
            os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))


_load_dotenv(os.path.join(os.getcwd(), ".env"))

from lago_agent_sdk import LagoSDK  # noqa: E402
from lago_agent_sdk.config import LagoConfig  # noqa: E402
from lago_agent_sdk.gateway.adapters import extract_cloudflare_log, resolve_subscription  # noqa: E402

_REQUIRED = ["CF_ACCOUNT_ID", "CF_GATEWAY_ID", "CF_LOGS_TOKEN", "LAGO_API_KEY"]
_missing = [name for name in _REQUIRED if not os.environ.get(name)]
if _missing:
    raise SystemExit(
        f"Missing required environment variable(s): {', '.join(_missing)}.\n"
        "Set them before starting the kernel, or put them in examples/.env — see the Setup cell above."
    )

CF_ACCOUNT_ID = os.environ["CF_ACCOUNT_ID"]
CF_GATEWAY_ID = os.environ["CF_GATEWAY_ID"]
CF_LOGS_TOKEN = os.environ["CF_LOGS_TOKEN"]
CF_GATEWAY_AUTH = os.environ.get("CF_GATEWAY_AUTH", "")
LAGO_API_KEY = os.environ["LAGO_API_KEY"]
LAGO_API_URL = os.environ.get("LAGO_API_URL", "https://api.getlago.com/api/v1")
LAGO_SUBSCRIPTION_ID = os.environ.get("LAGO_SUBSCRIPTION_ID", "cloudflare_gateway_demo_sub")
LAGO_VERIFY_SSL = os.environ.get("LAGO_VERIFY_SSL", "true").lower() != "false"
MISTRAL_API_KEY = os.environ.get("MISTRAL_API_KEY", "")

sdk = LagoSDK(
    api_key=LAGO_API_KEY,
    api_url=LAGO_API_URL,
    default_subscription_id=LAGO_SUBSCRIPTION_ID,
    config=LagoConfig(
        api_key=LAGO_API_KEY, api_url=LAGO_API_URL, pricing_mode="price", verify_ssl=LAGO_VERIFY_SSL,
        # Prices "workers-ai" calls from Cloudflare's own model catalog — the
        # real rate the gateway bills at, not a third party's guess. This is
        # just a credential declaration, not an eager fetch: Cloudflare's
        # catalog is only ever actually fetched lazily, on this session's
        # first real workers-ai call (see warm_pricing() below). Optional:
        # without it, Workers AI calls just fall back to token events.
        cloudflare_account_id=CF_ACCOUNT_ID, cloudflare_api_token=CF_GATEWAY_AUTH,
        # Mistral has no price table of its own — this resolves "-latest"
        # aliases (e.g. "mistral-small-latest") via Mistral's own /v1/models
        # to the dated id OpenRouter actually lists. Same as Cloudflare
        # above: declaring the key here doesn't fetch anything by itself —
        # it's fetched lazily on this session's first real Mistral call.
        # Optional: without it, an aliased Mistral call falls back to token
        # events instead.
        mistral_api_key=MISTRAL_API_KEY,
    ),
)
# Blocks until OpenRouter's table is fetched — closes the cold-start race for
# the very first call, for whichever native provider (anthropic/openai/
# mistral/gemini) that first call happens to use. Deliberately does NOT also
# force-fetch Cloudflare/Mistral above: both are credential-gated and
# provider-specific, and this demo (like most price-mode setups) may only
# ever call one of the three PROVIDER options below in a given run — eagerly
# hitting all their APIs regardless of which one gets used would be wasted
# work. So: whichever provider your first live call below actually uses,
# THAT one's table gets fetched lazily right then (and is cached for every
# call after) — only that very first call for a given provider can race a
# cold cache, and only if you picked workers-ai or mistral (never for
# anthropic/openai/gemini, which OpenRouter already warmed here).
sdk.warm_pricing()
print("SDK ready — billing to", LAGO_SUBSCRIPTION_ID)

## Part 1 — Backfill historic usage from Cloudflare

Fetch every log entry the gateway has recorded, and bill each one's real
Cloudflare-reported cost. No price lookup on our side — Cloudflare already
metered it.

`UNIFIED_BILLING = True` (the default here) bills every entry to
`LAGO_SUBSCRIPTION_ID`, ignoring any `cf-aig-metadata` attribution a call
might carry — the right choice when this gateway's traffic should all land on
one subscription. Set it `False` instead to respect real per-call
attribution and route each entry to whichever subscription it names,
falling back to `LAGO_SUBSCRIPTION_ID` only for entries with none — the right
choice when one gateway serves multiple customers/subscriptions.

In [2]:
def fetch_all_logs():
    entries, page = [], 1
    while True:
        body = requests.get(
            f"https://api.cloudflare.com/client/v4/accounts/{CF_ACCOUNT_ID}"
            f"/ai-gateway/gateways/{CF_GATEWAY_ID}/logs",
            headers={"Authorization": f"Bearer {CF_LOGS_TOKEN}"},
            params={"per_page": 50, "page": page},
            timeout=30,
        ).json()
        entries.extend(body["result"])
        if len(body["result"]) < 50 or len(entries) >= body["result_info"]["total_count"]:
            return entries
        page += 1


logs = fetch_all_logs()
print(f"fetched {len(logs)} real log entries from Cloudflare")

fetched 123 real log entries from Cloudflare


In [3]:
UNIFIED_BILLING = True

for entry in logs:
    usage = extract_cloudflare_log(entry)
    sub = LAGO_SUBSCRIPTION_ID if UNIFIED_BILLING else (resolve_subscription(entry) or LAGO_SUBSCRIPTION_ID)
    # transaction_id is unique across the whole ORG, not just this subscription —
    # always scope it by subscription. Without this, switching LAGO_SUBSCRIPTION_ID
    # to a second, different subscription later (unified or not) would have every
    # entry collide with the ids already used for the first one and silently never
    # land anywhere at all — this bit a real run: the first unified subscription's
    # ids blocked every entry from ever reaching a second one.
    event_id = f"unified_{sub}_{entry['id']}" if UNIFIED_BILLING else f"backfill_{sub}_{entry['id']}"
    sdk.emit(
        usage,
        subscription=sub,
        mode="price",
        usd_cost=entry.get("cost") or 0,  # Cloudflare's own metered price
        event_id=event_id,
    )

assert sdk.flush(timeout=30.0), "queue did not flush in time"
print(f"backfilled {len(logs)} entries")

backfilled 123 entries


## Part 2 — Live call through the gateway

Pick a provider. `workers-ai` needs no external key at all — Cloudflare bills
it directly. `anthropic`/`mistral` need their own key set as an env var.

In [4]:
PROMPT = "Tell me about getLago, the billing company - give as many details as you can find"

In [10]:
from anthropic import Anthropic
client = sdk.wrap(Anthropic(
        api_key=os.environ["ANTHROPIC_API_KEY"],
        base_url=f"https://gateway.ai.cloudflare.com/v1/{CF_ACCOUNT_ID}/{CF_GATEWAY_ID}/anthropic",
        default_headers={"cf-aig-authorization": f"Bearer {CF_GATEWAY_AUTH}"},
    ))
resp = client.messages.create(model="claude-sonnet-4-5", max_tokens=20000,
                                   messages=[{"role": "user", "content": PROMPT}])
text = resp.content[0].text

In [11]:
print(text)

# GetLago - Open-Source Billing Platform

## Overview
GetLago is an open-source billing and metering platform designed for product-led SaaS companies. It provides an alternative to proprietary billing solutions like Stripe Billing, Chargebee, and Recurly.

## Key Information

### Company Background
- **Founded**: 2021
- **Founders**: Anh-Tho Chuong and Raffi Sarkissian
- **Headquarters**: Paris, France (with remote-first culture)
- **Funding**: Raised significant seed funding from investors including Y Combinator (YC Winter 2022 batch), SignalFire, and others

### Core Product Features

**1. Usage-Based Billing**
- Real-time event ingestion and metering
- Supports complex pricing models (pay-as-you-go, tiered, graduated, package pricing)
- Aggregation capabilities for billing metrics

**2. Subscription Management**
- Handles recurring subscriptions
- Supports hybrid models (combining subscriptions + usage)
- Plan versioning and management

**3. Pricing Flexibility**
- Multiple charge m

In [12]:
from mistralai.client import Mistral
client = sdk.wrap(Mistral(
        api_key=os.environ["MISTRAL_API_KEY"],
        server_url=f"https://gateway.ai.cloudflare.com/v1/{CF_ACCOUNT_ID}/{CF_GATEWAY_ID}/mistral",
    ))
resp = client.chat.complete(model="mistral-small-latest",
                                 messages=[{"role": "user", "content": PROMPT}],
                                 http_headers={"cf-aig-authorization": f"Bearer {CF_GATEWAY_AUTH}"})
text = resp.choices[0].message.content
print(text)

lago pricing failed: no price for provider='mistral' model='mistral-small-latest' api='native'


**GetLago** is an open-source **usage-based billing** and **metering** platform designed to help SaaS companies implement **pay-as-you-go** pricing models efficiently. It provides developers with the tools to track customer usage, apply custom pricing rules, and generate invoices—all while integrating seamlessly with existing billing systems.

Here’s a detailed breakdown of **GetLago**, including its features, architecture, pricing, integrations, and more:

---

## **1. Overview & Key Features**
GetLago is built to solve the challenges of **usage-based billing**, which is becoming increasingly popular among SaaS companies (e.g., AWS, Stripe, Datadog). Key features include:

### **🔹 Core Features**
✅ **Metering & Usage Tracking**
- Tracks API calls, feature usage, storage, compute time, etc.
- Supports **real-time** and **batch** event ingestion.
- Customizable **metric definitions** (e.g., "API requests," "database queries").

✅ **Pricing & Billing Engine**
- Supports **tiered pricing*

In [ ]:
from openai import OpenAI
client = sdk.wrap(OpenAI(
        api_key=CF_GATEWAY_AUTH,
        base_url=f"https://gateway.ai.cloudflare.com/v1/{CF_ACCOUNT_ID}/{CF_GATEWAY_ID}/compat",
    ))
resp = client.chat.completions.create(
        model="workers-ai/@cf/meta/llama-3.3-70b-instruct-fp8-fast",
        messages=[{"role": "user", "content": PROMPT}],
    )
text = resp.choices[0].message.content

print(text)